# Libraries

In [ ]:
# 0. Clean slate — remove existing copies before reinstalling
!pip uninstall -y torch torchvision torchaudio torchao

!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 torchao==0.16.0 --index-url https://download.pytorch.org/whl/cu128

!pip install -q "cuda-core==0.3.*" "numba-cuda[cu12]<0.23.0,>=0.22.2" "numba<0.62.0,>=0.60.0"

!pip install -q llmcompressor==0.10.0.2

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 916.8/916.9 MB 193.1 MB/s eta 0:00:0100:01

In [ ]:
import random
import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)

from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

from huggingface_hub import (
    login,
    create_repo,
    upload_folder,
)

from kaggle_secrets import UserSecretsClient
from datasets import load_dataset, Dataset
import json

# Global Variables

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
OUTPUT_DIR = "/kaggle/working/Qwen3-4B-Instruct-2507-AWQ-4bit"
REPO_ID = f"EdgeCompress01/Qwen3-4B-Instruct-2507-AWQ-4bit-2026"
SEED = 42

# Functions

In [ ]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")


def format_chat(batch):
    texts = [
        tokenizer.apply_chat_template(
            text,
            tokenize=False,
            add_generation_prompt=False
        )
        for text in batch["text"]
    ]
    return {"text": texts}

# Set Seed

In [ ]:
set_reproducibility(SEED)

# Huggingface Login

In [ ]:
# --- 3. HUGGING FACE AUTHENTICATION ---
print("Logging into Hugging Face...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# Load Model & Tokenizer

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype="bfloat16"
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

# AWQ Quantization

**Configurations**

In [ ]:
recipe = [
    AWQModifier(
        ignore=["lm_head", "embed_tokens"], 
        scheme="W4A16",
        targets=["Linear"]
    )
]

**Calibration Dataset**

In [ ]:
dataset = load_dataset("monology/pile-uncopyrighted-parquet", split="train", streaming=True).shuffle(seed=SEED, buffer_size=1000)

samples = Dataset.from_dict({"text": [[{"role": "user", "content": s["text"]}] for s in dataset.take(64)]})

formatted_dataset = samples.map(format_chat, batched=True)

**Apply quantization**

In [ ]:
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=formatted_dataset,
    text_column="text",
    recipe=recipe,
    max_seq_length=2048,
    num_calibration_samples=64
)

In [ ]:
model.config._name_or_path = ""

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully")

# PUSH TO HUGGING FACE

In [ ]:
# create repo in organization
create_repo(REPO_ID, repo_type="model", exist_ok=True)

# upload to organization repo
upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=OUTPUT_DIR
)

print("Upload complete to organization!")